# Installing Starter Code


In [2]:
!pip install -q git+https://github.com/zhouy185/rosa-starter.git

import numpy as np
from starter import ZONES, TIME_BLOCKS, COSTS, PROMISE, delivery_times

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
times = delivery_times("Far West", "Fri/Sat eve", 45)
print(len(times), "orders; average delivery time", round(times.mean(), 1),
"minutes")

210 orders; average delivery time 43.2 minutes


In [4]:
# Part 1. a)

def late_orders(zone, time_block, promise, seed=1):

  if zone == "all":
    zone = ZONES
  else:
    zone = [zone]

  if time_block == "all":
    time_block = TIME_BLOCKS
  else:
    time_block = [time_block]

  all_times = []

  # Generate delivery times for each zone and time block
  for i in zone:
    for j in time_block:
      times = delivery_times(i, j, promise, seed=seed)
      all_times.append(times)

  times = np.concatenate(all_times)
  result = ((times > promise).mean()*100)
  return(result)


print(late_orders("Far West", "Fri/Sat eve", 45), "%")
print(late_orders("all", "Lunch", 45), "%")
print(late_orders("Far West", "all", 45), "%")
print(late_orders("all", "all", 45), "%")

print(f"{late_orders('Far West', 'Fri/Sat eve', 45):.1f}%")

36.666666666666664 %
3.3333333333333335 %
15.606936416184972 %
8.291873963515755 %
36.7%


In [5]:
# Part 1. b)

rankings = []
for i in ZONES:
  for j in TIME_BLOCKS:
    rate = late_orders(i, j, 45)
    rankings.append((i, j, rate))

rankings = sorted(rankings, key=lambda x: x[2], reverse=True)

for i, j, rate in rankings:
  print(f"{i}, {j}: {rate:.1f}%")

Far West, Fri/Sat eve: 36.7%
North, Fri/Sat eve: 16.7%
Far West, Weekday eve: 11.2%
Central, Fri/Sat eve: 6.8%
North, Weekday eve: 6.0%
Far West, Other: 4.7%
Central, Weekday eve: 4.2%
North, Lunch: 3.7%
Central, Lunch: 3.5%
Central, Other: 2.6%
North, Other: 2.5%
Far West, Lunch: 2.5%


In [6]:
# Part 1. c).

def average_delivery_time(zone, time_block, promise, seed=1):

  if zone == "all":
    zone = ZONES
  else:
    zone = [zone]

  if time_block == "all":
    time_block = TIME_BLOCKS
  else:
    time_block = [time_block]

  all_times = []

  # Generate delivery times for each zone and time block
  for i in zone:
    for j in time_block:
      times = delivery_times(i, j, promise, seed=seed)
      all_times.append(times)

  times = np.concatenate(all_times)
  result = times.mean()

  return result

  # testing
print(f"{average_delivery_time('Far West', 'Fri/Sat eve', 45):.1f} minutes")
print(f"{average_delivery_time('all', 'Lunch', 45):.1f} minutes")
print(f"{average_delivery_time('Far West', 'all', 45):.1f} minutes")
print(f"{average_delivery_time('all', 'all', 45):.1f} minutes")

42.9 minutes
28.2 minutes
35.3 minutes
31.2 minutes


In [7]:
# Part 1. d)

rankings = []

for i in ZONES:
  for j in TIME_BLOCKS:
    rate = average_delivery_time(i, j, 45)
    rankings.append((i, j, rate))

# Sort from longest to shortest average delivery time
rankings = sorted(rankings, key=lambda x: x[2], reverse=True)

for i, j, rate in rankings:
  print(f"{i}, {j}: {rate:.1f} minutes")


Far West, Fri/Sat eve: 42.9 minutes
North, Fri/Sat eve: 36.8 minutes
Far West, Weekday eve: 35.0 minutes
Central, Fri/Sat eve: 32.1 minutes
North, Weekday eve: 31.4 minutes
Far West, Lunch: 30.6 minutes
Far West, Other: 30.0 minutes
Central, Weekday eve: 29.0 minutes
North, Lunch: 28.7 minutes
North, Other: 27.7 minutes
Central, Lunch: 26.8 minutes
Central, Other: 26.0 minutes


# Part 1. e)
The late rate is more relevant to Rosa's decisions because it shows how often orders are actually late compared to the 45-minute promise. Rosa's costs come from late orders. Each late order triggers a refund and churn, so the late rate means revenue lost. Average delivery time doesn't effect revenue. For example, Far West, Fri/Sat eve averages 42.9 minutes, which is under the 45-minute promise, yet 36.7% of its orders are late. The average hides the problem because average delivery time does not directly show how many orders were late. Also, Far West, Lunch ranks 6th by average time (30.6 min) but last by late rate (2.5%). The two measures rank things differently, and only the late rate matches Rosa's costs.

Additionally, an average could be below 45 minutes even if some orders were still late. Since Rosa's promise is 45 minutes, I think the late rate is more useful for deciding where there is a problem. The average delivery time is still useful because it shows which areas and time blocks generally take longer, but the late rate gives a clearer idea of whether Rosa is meeting her promise to customers.

In [8]:
# Part 2. a)

def late_order_cost(costs):
  refund = costs["refund"]
  churn = costs["churn_orders"]
  margin = costs["margin"]

  # Total cost of a late order = refund + lost profit from churn
  total_cost = refund + (churn * margin)

  return total_cost

print("Cost per late order:", late_order_cost(COSTS))

Cost per late order: 26.2


In [13]:
# Part 2. b)

def best_promise(zone, time_block, promises, costs):

  best = None
  best_profit = None

  # Calculate the cost of each late order
  late_cost = late_order_cost(costs)

  # Test each promised delivery time
  for promise in promises:
    times = delivery_times(zone, time_block, promise, seed=1)

    orders = len(times)
    late = (times > promise).sum()

    profit = orders * costs["margin"]
    late_costs = late * late_cost


    # Net profit = profit from orders - cost of late orders
    net_profit = profit - late_costs

    if best_profit is None or net_profit > best_profit:
      best_profit = net_profit
      best = promise

  return best, best_profit

# Test promises
promises = range(25, 75, 5)

result = best_promise(
  "Far West",
  "Fri/Sat eve",
  promises,
  COSTS
)

print("Best promise:", result[0], "minutes")
print("Net profit:", result[1])

Best promise: 55 minutes
Net profit: 1212.4


In [14]:
# Test that runs every pair - Reccomendations for Rosa
promises = range(25, 75, 5)

for zone in ZONES:
  for block in TIME_BLOCKS:
    best, profit = best_promise(zone, block, promises, COSTS)
    print(f"{zone}, {block}: best promise {best} min, net profit {profit:.1f}")

Central, Lunch: best promise 45 min, net profit 2104.2
Central, Weekday eve: best promise 45 min, net profit 2603.2
Central, Fri/Sat eve: best promise 50 min, net profit 1399.0
Central, Other: best promise 45 min, net profit 1579.0
North, Lunch: best promise 45 min, net profit 1526.6
North, Weekday eve: best promise 45 min, net profit 1857.0
North, Fri/Sat eve: best promise 50 min, net profit 996.4
North, Other: best promise 45 min, net profit 1335.2
Far West, Lunch: best promise 45 min, net profit 1001.4
Far West, Weekday eve: best promise 50 min, net profit 1210.8
Far West, Fri/Sat eve: best promise 55 min, net profit 1212.4
Far West, Other: best promise 45 min, net profit 1492.2


# Additional Analysis + Reccomendations

Rosa should keep the 45 minute delivery promise for most zones and time blocks, however, I think she should use longer promises in the four situations with the highest late order rates. These are Far West on Fri/Sat evenings, North on Fri/Sat evenings, Far West on weekday evenings and Central on Fri/Sat evenings. A longer promise can reduce late orders and the costs that come with them. For other situations, the late order rates are lower so keeping the 45 minute promise makes more sense because a longer promise could reduce the number of customers who order.

The biggest improvement is for Far West on Fri/Sat evenings, where using a longer promise greatly improves net profit. North and Central on Fri/Sat evenings also benefit from longer promises, despite the improvement being smaller. However, these results are based on the simulator and one seed, so the results could be different in a real week. The recommendation also depends on the values in COSTS, which is why the Streamlit app allows Rosa to change the refund, churn and profit margin values and recalculate the best promise.

# Part 2. b).

I first tested promised delivery times from 5 to 50 minutes in 5-minute steps. The best promise was 50 minutes, but since 50 was the largest value I tested, I thought the best promise could be higher. When the best result is at the edge of the range, it makes sense to increase the range and test longer promised times.

When I increased the range, I found that the longer promises gave better results, so I changed the range to 25 to 75 minutes in 5-minute steps. I chose this range because promises that are too short can result in a large number of late orders and higher costs, while promises that are too long can reduce the number of customers who order. Testing from 25 to 75 minutes gives a wider range of realistic promised delivery times and makes it less likely that the best promise will be at the edge of the range.

My final range is:

promises = range(25, 75, 5)

I then used this range to compare the net profit for each promised delivery time and choose the promise with the highest net profit.

In [15]:
# Net profit for each promise, to check the range is wide enough
# This was strongly suggested by Claude because when you run it,
# you'll see that profit is very negative at short promises, peaks at 55,
# and then falls.

late_cost = late_order_cost(COSTS)

for promise in range(5, 81, 5):
  times = delivery_times("Far West", "Fri/Sat eve", promise, seed=1)
  orders = len(times)
  late = (times > promise).sum()
  net_profit = orders * COSTS["margin"] - late * late_cost
  print(f"{promise} min: {orders} orders, {late} late, net profit {net_profit:.1f}")

5 min: 242 orders, 242 late, net profit -4162.4
10 min: 242 orders, 242 late, net profit -4162.4
15 min: 241 orders, 241 late, net profit -4145.2
20 min: 241 orders, 239 late, net profit -4092.8
25 min: 239 orders, 233 late, net profit -3953.6
30 min: 237 orders, 218 late, net profit -3578.6
35 min: 232 orders, 179 late, net profit -2601.8
40 min: 224 orders, 133 late, net profit -1468.6
45 min: 210 orders, 77 late, net profit -127.4
50 min: 188 orders, 33 late, net profit 827.4
55 min: 158 orders, 8 late, net profit 1212.4
60 min: 121 orders, 2 late, net profit 1036.6
65 min: 84 orders, 0 late, net profit 756.0
70 min: 54 orders, 0 late, net profit 486.0
75 min: 32 orders, 0 late, net profit 288.0
80 min: 18 orders, 0 late, net profit 162.0


# Link to Github repo
https://github.com/ThePurpleMan16/rosa-app



#Link to streamlit app

https://rosa-app-gzfqqvvqcmoyra2j2elabw.streamlit.app/